In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv("train_labels.csv")
val_df = pd.read_csv("val_labels.csv")
test_df = pd.read_csv("test_labels.csv")

train_texts = train_df["text"].tolist()
val_texts = val_df["text"].tolist()
test_texts = test_df["text"].tolist()


In [3]:
unique_train_texts = sorted(set(train_texts))
sentence_to_id = {text: i for i, text in enumerate(unique_train_texts)}

UNK_ID = len(sentence_to_id)

In [4]:
def one_hot_encode(text, sentence_to_id, unk_id=None):
    if text in sentence_to_id:
        idx = sentence_to_id[text]
        size = len(sentence_to_id) if unk_id is None else len(sentence_to_id) + 1
    else:
        if unk_id is None:
            raise ValueError(f"Unseen text found: {text}")
        idx = unk_id
        size = len(sentence_to_id) + 1

    vec = np.zeros(size, dtype=np.float32)
    vec[idx] = 1.0
    return vec

X_train_onehot = np.array([one_hot_encode(t, sentence_to_id, UNK_ID) for t in train_texts])
X_val_onehot = np.array([one_hot_encode(t, sentence_to_id, UNK_ID) for t in val_texts])
X_test_onehot = np.array([one_hot_encode(t, sentence_to_id, UNK_ID) for t in test_texts])

print("Train one-hot shape:", X_train_onehot.shape)
print("Val one-hot shape:", X_val_onehot.shape)
print("Test one-hot shape:", X_test_onehot.shape)

print("Example text:", train_texts[0])
print("Example one-hot:", X_train_onehot[0])

Train one-hot shape: (70, 60)
Val one-hot shape: (15, 60)
Test one-hot shape: (15, 60)
Example text: There are one die showing five.
Example one-hot: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [5]:
print(X_train_onehot[0].sum())
print(X_val_onehot[0].sum())
print(X_test_onehot[0].sum())

1.0
1.0
1.0


In [6]:
unk_count_val = sum(np.argmax(v) == UNK_ID for v in X_val_onehot)
unk_count_test = sum(np.argmax(v) == UNK_ID for v in X_test_onehot)

print("UNK in val:", unk_count_val)
print("UNK in test:", unk_count_test)

UNK in val: 10
UNK in test: 10


In [7]:
# TF-DIF
from sklearn.feature_extraction.text import TfidfVectorizer

train_texts = train_df["text"].tolist()
val_texts = val_df["text"].tolist()
test_texts = test_df["text"].tolist()

In [8]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_val_tfidf = tfidf_vectorizer.transform(val_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

In [9]:
print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Val TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

print("Number of words in vocabulary:", len(tfidf_vectorizer.vocabulary_))

Train TF-IDF shape: (70, 24)
Val TF-IDF shape: (15, 24)
Test TF-IDF shape: (15, 24)
Number of words in vocabulary: 24


In [10]:
feature_names = tfidf_vectorizer.get_feature_names()

print("Example text:", train_texts[0])

example_vector = X_train_tfidf[0].toarray()[0]

nonzero_indices = example_vector.nonzero()[0]

print("\nNon-zero TF-IDF values:")
for idx in nonzero_indices:
    print(feature_names[idx], "->", example_vector[idx])

Example text: There are one die showing five.

Non-zero TF-IDF values:
are -> 0.3388200198557353
die -> 0.4055929393635059
five -> 0.48212766343768315
one -> 0.3173450326490968
showing -> 0.44019337119298735
there -> 0.44019337119298735


In [11]:
print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Val TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)
print("Number of words in vocabulary:", len(tfidf_vectorizer.vocabulary_))

Train TF-IDF shape: (70, 24)
Val TF-IDF shape: (15, 24)
Test TF-IDF shape: (15, 24)
Number of words in vocabulary: 24


In [12]:
from sentence_transformers import SentenceTransformer

D:\Anaconda\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [13]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

X_train_sbert = sbert_model.encode(train_texts, convert_to_numpy=True)
X_val_sbert = sbert_model.encode(val_texts, convert_to_numpy=True)
X_test_sbert = sbert_model.encode(test_texts, convert_to_numpy=True)

print("Train SBERT shape:", X_train_sbert.shape)
print("Val SBERT shape:", X_val_sbert.shape)
print("Test SBERT shape:", X_test_sbert.shape)

print("\nExample text:", train_texts[0])
print("Example SBERT vector (first 10 values):", X_train_sbert[0][:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\Anaconda\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ll\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Train SBERT shape: (70, 384)
Val SBERT shape: (15, 384)
Test SBERT shape: (15, 384)

Example text: There are one die showing five.
Example SBERT vector (first 10 values): [ 0.00838555  0.0017225   0.00573912 -0.06765112  0.00191381  0.03956684
  0.03731402 -0.05268652  0.07273647  0.04609537]
